# Gene Region Retrieval

This notebook extracts the human gene symbols from the macrophage-associated protein list and retrieves their genomic coordinates using the Ensembl REST API.

In [1]:
%pip install openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
import re
from pathlib import Path

In [3]:
raw_gene_list_dir = Path("../data/raw/gene_list")
interim_gene_list_dir = Path("../data/interim/gene_list")
ensembl_out_dir = Path("../data/interim/ensembl")

interim_gene_list_dir.mkdir(parents=True, exist_ok=True)
ensembl_out_dir.mkdir(parents=True, exist_ok=True)

protein_file = raw_gene_list_dir / "macrophage_associated_proteins.xlsx"

human_gene_symbols_file = interim_gene_list_dir / "human_gene_symbols.csv"
gene_symbols_review_file = interim_gene_list_dir / "gene_symbols_for_review.csv"

grch38_regions_file = ensembl_out_dir / "gene_regions_grch38.csv"
grch37_regions_file = ensembl_out_dir / "gene_regions_grch37.csv"

In [4]:
protein_df = pd.read_excel(
    protein_file,
    sheet_name="Macrophage_protein_table",
    header=3
)

protein_df.columns = protein_df.columns.str.strip()

protein_df.head()

,Rat gene,Human gene,Protein,Direction in rat infarcted vs healthy,log2FC,Fold-change,q-value,Priority,Macrophage relevance,Macrophage-linked process,Rationale / why included
0,Csf1r,CSF1R,M-CSF receptor / CSF1R,Up,2.172,4.506477,0.005159,Very high,Direct/strong,Macrophage lineage receptor signaling,Central receptor for CSF1-dependent macrophage...
1,Csf1,CSF1,M-CSF / macrophage colony-stimulating factor 1,Up,1.805,3.494292,0.021778,Very high,Direct/strong,"Macrophage survival, differentiation and proli...",Core macrophage lineage cytokine; supports mon...
2,Cd14,CD14,Monocyte differentiation antigen CD14,Up,0.814,1.758079,0.022382,Very high,Direct/strong,Monocyte/macrophage innate immune sensing,Canonical monocyte/macrophage marker involved ...
3,Itgam,ITGAM,Integrin alpha M / CD11b,Up,1.296,2.455471,0.024961,Very high,Direct/strong,"Myeloid adhesion, migration and phagocytosis",Classic myeloid/macrophage integrin; supports ...
4,Itgb2,ITGB2,Integrin beta-2 / CD18,Up,2.175,4.515858,0.023378,High,Direct/strong,Leukocyte adhesion and trafficking,Pairs with myeloid integrins such as CD11b/CD1...


In [5]:
protein_df.columns.tolist()

['Rat gene',
 'Human gene',
 'Protein',
 'Direction in rat infarcted vs healthy',
 'log2FC',
 'Fold-change',
 'q-value',
 'Priority',
 'Macrophage relevance',
 'Macrophage-linked process',
 'Rationale / why included']

In [6]:
human_gene_column = "Human gene"

if human_gene_column not in protein_df.columns:
    raise ValueError(
        f"Expected column '{human_gene_column}' was not found. "
        f"Available columns are: {protein_df.columns.tolist()}"
    )

In [7]:
gene_input_df = (
    protein_df[[human_gene_column]]
    .rename(columns={human_gene_column: "gene_symbol"})
    .dropna()
    .copy()
)

gene_input_df["gene_symbol"] = (
    gene_input_df["gene_symbol"]
    .astype(str)
    .str.strip()
)

gene_input_df = gene_input_df.drop_duplicates().reset_index(drop=True)

gene_input_df.head()

,gene_symbol
0,CSF1R
1,CSF1
2,CD14
3,ITGAM
4,ITGB2


In [8]:
def looks_like_gene_symbol(value):
    """
    Basic check for values that look like standard gene symbols.
    This does not prove that the symbol is official, but it helps flag
    descriptive entries that are not directly usable for Ensembl lookup.
    """
    if pd.isna(value):
        return False

    value = str(value).strip()

    if value == "":
        return False

    if any(token in value.lower() for token in ["ortholog", "class", "family"]):
        return False

    if any(separator in value for separator in [",", ";", "/"]):
        return False

    return bool(re.match(r"^[A-Za-z0-9][A-Za-z0-9.-]*$", value))


gene_input_df["needs_manual_review"] = ~gene_input_df["gene_symbol"].apply(looks_like_gene_symbol)

gene_symbols_for_lookup = (
    gene_input_df.loc[~gene_input_df["needs_manual_review"], ["gene_symbol"]]
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

gene_symbols_needing_review = (
    gene_input_df.loc[gene_input_df["needs_manual_review"]]
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

gene_symbols_for_lookup.head(), gene_symbols_needing_review.head()

(  gene_symbol
 0        APOE
 1         B2M
 2        C1QA
 3        C1QB
 4        C1QC,
             gene_symbol  needs_manual_review
 0  HLA class I ortholog                 True)

In [9]:
gene_symbols_for_lookup.to_csv(human_gene_symbols_file, index=False)
gene_symbols_needing_review.to_csv(gene_symbols_review_file, index=False)

In [10]:
species = "homo_sapiens"

ensembl_servers = {
    "GRCh38": "https://rest.ensembl.org",
    "GRCh37": "https://grch37.rest.ensembl.org",
}

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
}

In [11]:
def lookup_gene_symbols(gene_symbols, server, species="homo_sapiens"):
    """
    Query Ensembl REST lookup/symbol endpoint for multiple gene symbols.
    Returns the decoded JSON response.
    """
    endpoint = f"/lookup/symbol/{species}"
    payload = {
        "symbols": gene_symbols
    }

    response = requests.post(
        server + endpoint,
        headers=headers,
        json=payload,
        timeout=60
    )

    if not response.ok:
        response.raise_for_status()

    return response.json()

In [12]:
gene_symbols = gene_symbols_for_lookup["gene_symbol"].tolist()

grch38_response = lookup_gene_symbols(
    gene_symbols=gene_symbols,
    server=ensembl_servers["GRCh38"],
    species=species
)

grch37_response = lookup_gene_symbols(
    gene_symbols=gene_symbols,
    server=ensembl_servers["GRCh37"],
    species=species
)

list(grch38_response.keys())[:5]

['CAPG', 'LBP', 'FN1', 'CTSZ', 'CCL2']

In [13]:
def ensembl_response_to_dataframe(response_json, input_symbols):
    rows = []

    for input_symbol in input_symbols:
        gene_data = response_json.get(input_symbol)

        if gene_data is None:
            rows.append({
                "input_gene_symbol": input_symbol,
                "lookup_status": "not_found",
                "ensembl_gene_id": None,
                "official_gene_symbol": None,
                "display_name": None,
                "species": species,
                "chromosome": None,
                "gene_start": None,
                "gene_end": None,
                "strand": None,
                "strand_symbol": None,
                "assembly_name": None,
                "biotype": None,
                "object_type": None,
                "source": "Ensembl REST",
            })
            continue

        strand_value = gene_data.get("strand")
        strand_symbol = "+" if strand_value == 1 else "-" if strand_value == -1 else None

        rows.append({
            "input_gene_symbol": input_symbol,
            "lookup_status": "found",
            "ensembl_gene_id": gene_data.get("id"),
            "official_gene_symbol": gene_data.get("display_name"),
            "display_name": gene_data.get("display_name"),
            "species": species,
            "chromosome": gene_data.get("seq_region_name"),
            "gene_start": gene_data.get("start"),
            "gene_end": gene_data.get("end"),
            "strand": strand_value,
            "strand_symbol": strand_symbol,
            "assembly_name": gene_data.get("assembly_name"),
            "biotype": gene_data.get("biotype"),
            "object_type": gene_data.get("object_type"),
            "source": "Ensembl REST",
        })

    return pd.DataFrame(rows)

In [14]:
grch38_gene_df = ensembl_response_to_dataframe(
    response_json=grch38_response,
    input_symbols=gene_symbols
)

grch37_gene_df = ensembl_response_to_dataframe(
    response_json=grch37_response,
    input_symbols=gene_symbols
)

In [15]:
flanking_bp = 50_000

def add_region_columns(gene_df, flanking_bp=50_000):
    region_df = gene_df.copy()

    region_df["flanking_bp"] = flanking_bp

    region_df["gene_start"] = pd.to_numeric(region_df["gene_start"], errors="coerce")
    region_df["gene_end"] = pd.to_numeric(region_df["gene_end"], errors="coerce")

    region_df["region_start"] = (region_df["gene_start"] - flanking_bp).clip(lower=1)
    region_df["region_end"] = region_df["gene_end"] + flanking_bp

    region_df["region_start"] = region_df["region_start"].astype("Int64")
    region_df["region_end"] = region_df["region_end"].astype("Int64")
    region_df["gene_start"] = region_df["gene_start"].astype("Int64")
    region_df["gene_end"] = region_df["gene_end"].astype("Int64")

    return region_df

In [16]:
def add_strand_aware_boundaries(region_df):
    region_df = region_df.copy()

    region_df["upstream_start"] = pd.NA
    region_df["upstream_end"] = pd.NA
    region_df["downstream_start"] = pd.NA
    region_df["downstream_end"] = pd.NA

    positive = region_df["strand"] == 1
    negative = region_df["strand"] == -1

    # Positive strand:
    # upstream is before gene start, downstream is after gene end
    region_df.loc[positive, "upstream_start"] = region_df.loc[positive, "region_start"]
    region_df.loc[positive, "upstream_end"] = region_df.loc[positive, "gene_start"] - 1
    region_df.loc[positive, "downstream_start"] = region_df.loc[positive, "gene_end"] + 1
    region_df.loc[positive, "downstream_end"] = region_df.loc[positive, "region_end"]

    # Negative strand:
    # biological upstream/downstream are reversed
    region_df.loc[negative, "upstream_start"] = region_df.loc[negative, "gene_end"] + 1
    region_df.loc[negative, "upstream_end"] = region_df.loc[negative, "region_end"]
    region_df.loc[negative, "downstream_start"] = region_df.loc[negative, "region_start"]
    region_df.loc[negative, "downstream_end"] = region_df.loc[negative, "gene_start"] - 1

    boundary_columns = [
        "upstream_start",
        "upstream_end",
        "downstream_start",
        "downstream_end",
    ]

    for column in boundary_columns:
        region_df[column] = region_df[column].astype("Int64")

    return region_df

In [17]:
grch38_regions = add_region_columns(grch38_gene_df, flanking_bp=flanking_bp)
grch38_regions = add_strand_aware_boundaries(grch38_regions)

grch37_regions = add_region_columns(grch37_gene_df, flanking_bp=flanking_bp)
grch37_regions = add_strand_aware_boundaries(grch37_regions)

grch38_regions.head()

,input_gene_symbol,lookup_status,ensembl_gene_id,official_gene_symbol,display_name,species,chromosome,gene_start,gene_end,strand,...,biotype,object_type,source,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end
0,APOE,found,ENSG00000130203,APOE,APOE,homo_sapiens,19,44903787,44909396,1,...,protein_coding,Gene,Ensembl REST,50000,44853787,44959396,44853787,44903786,44909397,44959396
1,B2M,found,ENSG00000166710,B2M,B2M,homo_sapiens,15,44711253,44718851,1,...,protein_coding,Gene,Ensembl REST,50000,44661253,44768851,44661253,44711252,44718852,44768851
2,C1QA,found,ENSG00000173372,C1QA,C1QA,homo_sapiens,1,22635057,22639681,1,...,protein_coding,Gene,Ensembl REST,50000,22585057,22689681,22585057,22635056,22639682,22689681
3,C1QB,found,ENSG00000173369,C1QB,C1QB,homo_sapiens,1,22652713,22661637,1,...,protein_coding,Gene,Ensembl REST,50000,22602713,22711637,22602713,22652712,22661638,22711637
4,C1QC,found,ENSG00000159189,C1QC,C1QC,homo_sapiens,1,22642558,22648110,1,...,protein_coding,Gene,Ensembl REST,50000,22592558,22698110,22592558,22642557,22648111,22698110


In [18]:
region_columns = [
    "input_gene_symbol",
    "official_gene_symbol",
    "ensembl_gene_id",
    "lookup_status",
    "species",
    "chromosome",
    "gene_start",
    "gene_end",
    "strand",
    "strand_symbol",
    "assembly_name",
    "flanking_bp",
    "region_start",
    "region_end",
    "upstream_start",
    "upstream_end",
    "downstream_start",
    "downstream_end",
    "biotype",
    "object_type",
    "source",
]

grch38_regions = grch38_regions[region_columns]
grch37_regions = grch37_regions[region_columns]

grch38_regions.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,...,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,...,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,...,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,...,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,...,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,...,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST


In [19]:
grch38_regions.to_csv(grch38_regions_file, index=False)
grch37_regions.to_csv(grch37_regions_file, index=False)

In [20]:
grch38_lookup_counts = grch38_regions["lookup_status"].value_counts().to_dict()
grch37_lookup_counts = grch37_regions["lookup_status"].value_counts().to_dict()

summary = {
    "flanking_bp": flanking_bp,
    "input_gene_symbols_total": len(gene_input_df),
    "gene_symbols_for_lookup": len(gene_symbols_for_lookup),
    "gene_symbols_for_review": len(gene_symbols_needing_review),
    "grch38_found": grch38_lookup_counts.get("found", 0),
    "grch38_not_found": grch38_lookup_counts.get("not_found", 0),
    "grch37_found": grch37_lookup_counts.get("found", 0),
    "grch37_not_found": grch37_lookup_counts.get("not_found", 0),
}

summary

{'flanking_bp': 50000,
 'input_gene_symbols_total': 52,
 'gene_symbols_for_lookup': 51,
 'gene_symbols_for_review': 1,
 'grch38_found': 51,
 'grch38_not_found': 0,
 'grch37_found': 51,
 'grch37_not_found': 0}